# Lab 4.2 · Polynomial Inner Products and Approximation

Separate polynomial coefficients from function values, derive the Gram matrix, and project a polynomial using the lecture’s integral inner product.

Wooseok Ha, MAS110 Lecture 4 (Fall 2026), lec04.pdf: printed slides 7–11, 18, 29, 34–39 (PDF pages 9–13, 21, 33, 39–44).

Maintained in [MAS110 Practice Sessions](https://github.com/statchan1106/mas110-practice-sessions). Course companion to [Foundations of LADS](https://github.com/kyunghyuncho/Foundations_of_LADS) by Wanmo Kang and Kyunghyun Cho.

Run all cells in order. NumPy handles the calculations; the final plot uses Matplotlib. Both are available in Colab.

A polynomial is a vector in a finite-dimensional function space. Its coefficient array is a coordinate representation, not the definition of its length. We use the lecture’s inner product ⟨f,g⟩ = ∫₋₁¹ f(t)g(t) dt and start with 𝒫₂, the polynomials of degree at most two, including the zero polynomial.

## Notation

- **⟨u,v⟩; ‖u‖; u ⊥ v**: An inner product gives one real number, ‖u‖ = √⟨u,u⟩ is its induced norm, and u ⊥ v means ⟨u,v⟩ = 0. The lecture writes |u| for this norm. In Euclidean space the default is uᵀv and the norm is ‖u‖₂; for polynomials it is the stated integral norm.

- **φ; θ; θ̂**: This chapter uses φ for an angle and θ for model coefficients in Lab 4.4. The lecture sometimes uses θ for the angle too; the distinction here prevents mixing an angle with a coefficient vector. A hat marks a minimizing coefficient, not a unit vector.

- **A; G; Q; R; P**: A stores input columns in the Euclidean QR and least-squares labs. G denotes a Gram matrix of inner products, including the polynomial metric (which the lecture locally calls A). Q stores orthonormal columns, R stores their reconstruction coefficients, and P projects onto a subspace. These matrices have different jobs and shapes.

- **𝒫ₙ; B = (1,t,…,tⁿ)**: 𝒫ₙ has dimension n + 1. “Degree at most n” includes lower degrees and the zero polynomial; polynomials of degree exactly n do not form a vector space. B is an ordered basis, so order determines which coefficient goes with each term.

- **f(t) = c₀ + c₁t + c₂t²; [f]ᴮ = c**: The coordinate vector c = (c₀,c₁,c₂)ᵀ has three entries. The scalar t is the argument; f(t) is the value at that argument. This page uses t instead of the lecture’s occasional x to distinguish an argument from a coordinate vector.

- **Gᵢⱼ = ∫₋₁¹ tⁱ⁺ʲ dt, i,j = 0,…,n**: These indices label monomial degrees, starting at 0. Gᵢⱼ = 0 for odd i+j and 2/(i+j+1) for even i+j. The lecture calls this Gram matrix A locally; we call it G to keep A for the design matrix in 4.3–4.4.

- **⟨f,g⟩ = cᵀGd; ‖f‖² = cᵀGc**: c and d are the coordinates of f and g in B. The plain coordinate dot product cᵀd is generally different. For example, the coordinate arrays for 1 and t² have dot product 0, but their integral inner product is 2/3.

- **q₀,q₁,q₂; C = [[q₀]ᴮ | [q₁]ᴮ | [q₂]ᴮ]**: We label polynomial basis vectors by degree, beginning at 0; these correspond to the lecture’s w₁,w₂,w₃. C has coefficient columns. Orthonormality for this metric means CᵀGC = I₃, not CᵀC = I₃.

- **Legendre Pₖ(t); normalized qₖ(t)**: Standard Legendre polynomials satisfy Pₖ(1) = 1 and ∫₋₁¹ Pₖ² = 2/(2k+1). Multiplying by √((2k+1)/2) gives qₖ of unit integral norm. Endpoint normalization and unit-norm normalization are different.

- **NumPy ascending powers: [c₀,c₁,c₂]**: numpy.polynomial.polynomial uses increasing powers: [0,1] is t and [0,0,1] is t². The original notebook uses legacy np.polyval/np.polyint, whose arrays run in the opposite order. We use the modern ascending convention consistently and do not mix the two APIs.

## Key ideas

### Polynomial coordinates

The ordered monomial basis identifies a polynomial with a coefficient vector.

f(t) = c₀ + c₁t + c₂t²

Keep in mind: A coefficient is not a sampled function value.

### Integral inner product

Integrate the product of two functions over the specified interval.

⟨f,g⟩ = ∫₋₁¹ fg = cᵀGd

Keep in mind: Changing the interval or weight changes the geometry.

### Orthonormal polynomials

Basis functions have unit integral norms and zero cross inner products.

CᵀGC = I

Keep in mind: CᵀC checks a different metric.

## 1. Store coefficients in one fixed order

The modern polynomial API stores the constant first. Make the convention visible before doing any arithmetic.

**Predict first:** What polynomial does [1,0,0] represent here?

- Line 1, `import numpy as np`: Loads NumPy. Shape: module. Operation: np supplies coefficient arrays.

- Line 2, `from numpy.polynomial import polynomial as poly`: Selects the ascending-power API. Shape: module. Operation: poly.polyval(t,c) evaluates c₀+c₁t+….

- Line 3, `f = np.array([0., 1.])`: Stores f(t) = t. Shape: f: (2,). Operation: [0,1] has zero constant and linear coefficient 1.

- Line 4, `g = np.array([0., 0., 1.])`: Stores g(t) = t². Shape: g: (3,). Operation: [0,0,1] has quadratic coefficient 1.

In [ ]:
import numpy as np
from numpy.polynomial import polynomial as poly
f = np.array([0., 1.])
g = np.array([0., 0., 1.])

In [ ]:
print("f coefficients =", f, "g coefficients =", g)

**Read the result:** Missing higher coefficients are implicitly zero. Arrays may have different lengths when multiplication and integration use the polynomial API.

f(t) = t; g(t) = t²



## 2. Integrate the polynomial product

Multiply the functions, form an antiderivative, then evaluate the endpoints. This uses polynomial arithmetic rather than sampling a curve.

**Predict first:** Does a zero integral imply the product is zero at every t?

- Line 1, `def inner_poly(f, g):`: Defines the inner product on this interval. Shape: two coefficient arrays → scalar. Operation: The body runs when the function is called.

- Line 2, `    product = poly.polymul(f, g)`: Multiplies polynomial functions. Shape: polynomial coefficients → polynomial coefficients. Operation: For f=t and g=t², product=t³.

- Line 3, `    antiderivative = poly.polyint(product)`: Finds an antiderivative. Shape: degree d → degree d+1. Operation: The integration constant cancels in the endpoint difference.

- Line 4, `    return poly.polyval(1., antiderivative) - poly.polyval(-1., antiderivative)`: Evaluates the definite integral. Shape: two scalar evaluations → scalar. Operation: For t³ this is 1/4 − 1/4 = 0.

- Line 5, `dot_fg = inner_poly(f, g)`: Computes the cross inner product. Shape: scalar. Operation: dot_fg = 0, by odd symmetry.

- Line 6, `lengths = np.sqrt([inner_poly(f, f), inner_poly(g, g)])`: Computes integral norms. Shape: two scalars → lengths: (2,). Operation: The lengths are √(2/3) and √(2/5).

In [ ]:
def inner_poly(f, g):
    product = poly.polymul(f, g)
    antiderivative = poly.polyint(product)
    return poly.polyval(1., antiderivative) - poly.polyval(-1., antiderivative)
dot_fg = inner_poly(f, g)
lengths = np.sqrt([inner_poly(f, f), inner_poly(g, g)])

In [ ]:
print("<f,g> =", dot_fg, "integral norms =", lengths)

**Read the result:** Orthogonality describes the signed integral of fg, not a right angle between curves on the page.

⟨t,t²⟩ = 0; ‖t‖² = 2/3; ‖t²‖² = 2/5



## 3. Build the Gram matrix of the monomial basis

Each entry is the integral of a pair of basis functions. This is the metric acting on coefficient vectors.

**Predict first:** Why is the top-right entry 2/3 instead of 0?

- Line 1, `B = np.eye(3)`: Stores coefficient rows for 1, t, t². Shape: B: (3, 3). Operation: Here row i has the coefficient of tⁱ equal to 1.

- Line 2, `G = np.array([[inner_poly(bi, bj) for bj in B] for bi in B])`: Computes all basis-pair integrals. Shape: 3 × 3 scalar pairs → G: (3, 3). Operation: Even powers integrate to 2/(degree+1); odd powers to 0.

- Line 3, `coefficient_inner = np.array([0., 1., 0.]) @ G @ g`: Recomputes ⟨t,t²⟩ through coordinates. Shape: (3,) @ (3, 3) @ (3,) → scalar. Operation: The result is 0, agreeing with direct integration.

- Line 4, `contrast = np.array([B[0] @ B[2], inner_poly(B[0], B[2])])`: Contrasts coefficient and integral metrics. Shape: two scalars → contrast: (2,). Operation: For 1 and t², coefficient dot product is 0 but integral inner product is 2/3.

In [ ]:
B = np.eye(3)
G = np.array([[inner_poly(bi, bj) for bj in B] for bi in B])
coefficient_inner = np.array([0., 1., 0.]) @ G @ g
contrast = np.array([B[0] @ B[2], inner_poly(B[0], B[2])])

In [ ]:
print("G =", G, "<t,t^2> from coordinates =", coefficient_inner, "dot vs integral for 1,t^2 =", contrast, sep="\n")

**Read the result:** The identity matrix B stores basis coordinates; G stores inner products. They are not the same matrix.

⟨f,g⟩ = [f]ᴮᵀG[g]ᴮ



## 4. Subtract the constant component of t²

Normalize the first two monomials, then subtract both earlier orthonormal components from t². One component is zero, but it must still be accounted for.

**Predict first:** Why does subtracting 1/3 make t² perpendicular to the constant?

- Line 1, `q0 = B[0] / np.sqrt(2.)`: Normalizes the constant polynomial. Shape: (3,) → (3,). Operation: q₀(t) = 1/√2.

- Line 2, `q1 = B[1] / np.sqrt(2. / 3.)`: Normalizes the linear polynomial. Shape: (3,) → (3,). Operation: q₁(t) = √(3/2)t.

- Line 3, `w2 = B[2] - inner_poly(B[2], q0) * q0 - inner_poly(B[2], q1) * q1`: Removes every earlier component. Shape: three (3,) arrays → (3,). Operation: w₂(t) = t² − 1/3; its integral against 1 and t is zero.

- Line 4, `q2 = w2 / np.sqrt(inner_poly(w2, w2))`: Normalizes the remaining polynomial. Shape: (3,) / scalar → (3,). Operation: ‖w₂‖² = 8/45, so q₂ = √(45/8)(t²−1/3).

- Line 5, `C = np.column_stack((q0, q1, q2))`: Stores orthonormal coefficient columns. Shape: C: (3, 3). Operation: Each column represents one polynomial in the monomial basis.

- Line 6, `orthonormal_gram = C.T @ G @ C`: Checks the correct metric. Shape: (3, 3) @ (3, 3) @ (3, 3) → (3, 3). Operation: CᵀGC ≈ I₃; CᵀC is not the relevant check.

In [ ]:
q0 = B[0] / np.sqrt(2.)
q1 = B[1] / np.sqrt(2. / 3.)
w2 = B[2] - inner_poly(B[2], q0) * q0 - inner_poly(B[2], q1) * q1
q2 = w2 / np.sqrt(inner_poly(w2, w2))
C = np.column_stack((q0, q1, q2))
orthonormal_gram = C.T @ G @ C

In [ ]:
print("q0 =", q0, "q1 =", q1, "w2 =", w2, "q2 =", q2, "C.T @ G @ C =", orthonormal_gram, sep="\n")

**Read the result:** The three functions are orthonormal even though their ordinary coefficient dot products are not.

q₀ = 1/√2; q₁ = √(3/2)t; q₂ = √(45/8)(t²−1/3)



## 5. Project t² onto the polynomials of degree at most one

Use q₀ and q₁ only, because the approximation space is 𝒫₁. Orthogonality makes each coefficient independent.

**Predict first:** Why is the best line a constant here?

- Line 1, `target = B[2]`: Chooses the target t². Shape: target: (3,). Operation: Its coordinates are [0,0,1].

- Line 2, `projection = inner_poly(target, q0) * q0 + inner_poly(target, q1) * q1`: Keeps the two allowed orthonormal components. Shape: two (3,) arrays → (3,). Operation: The projection is [1/3,0,0], representing the constant 1/3.

- Line 3, `residual = target - projection`: Computes the residual polynomial. Shape: (3,) − (3,) → (3,). Operation: e(t) = t² − 1/3.

- Line 4, `orthogonality = np.array([inner_poly(residual, q0), inner_poly(residual, q1)])`: Checks orthogonality to the approximation space. Shape: two scalars → (2,). Operation: Both integrals are approximately zero.

- Line 5, `minimum_error = inner_poly(residual, residual)`: Computes integrated squared error. Shape: scalar. Operation: ∫₋₁¹ (t²−1/3)² dt = 8/45.

In [ ]:
target = B[2]
projection = inner_poly(target, q0) * q0 + inner_poly(target, q1) * q1
residual = target - projection
orthogonality = np.array([inner_poly(residual, q0), inner_poly(residual, q1)])
minimum_error = inner_poly(residual, residual)

In [ ]:
print("projection =", projection, "residual =", residual, "orthogonality =", orthogonality, "minimum squared error =", minimum_error, sep="\n")

**Read the result:** The closest line minimizes an integral, not pointwise error at every t. Its residual is perpendicular to every line in 𝒫₁.

p(t) = 1/3; min ∫₋₁¹ (t²−p(t))² dt = 8/45



## 6. Recover the same approximation from a matrix system

The metric-weighted normal equations are another form of the same projection. The metric G belongs in the system.

**Predict first:** What would go wrong if you replaced G by I₃?

- Line 1, `E = B[:2].T`: Stores coordinates of 1 and t as columns. Shape: E: (3, 2). Operation: The columns span the coefficient representation of 𝒫₁.

- Line 2, `H = E.T @ G @ E`: Builds the subspace Gram matrix. Shape: (2, 3) @ (3, 3) @ (3, 2) → (2, 2). Operation: H = diag(2,2/3).

- Line 3, `rhs = E.T @ G @ target`: Computes target inner products. Shape: (2, 3) @ (3, 3) @ (3,) → (2,). Operation: rhs = (2/3,0)ᵀ.

- Line 4, `alpha = np.linalg.solve(H, rhs)`: Solves for coordinates in (1,t). Shape: solve((2, 2), (2,)) → (2,). Operation: α = (1/3,0)ᵀ.

- Line 5, `projection_gram = E @ alpha`: Forms the full coefficient vector. Shape: (3, 2) @ (2,) → (3,). Operation: The result [1/3,0,0] agrees with the orthonormal projection.

In [ ]:
E = B[:2].T
H = E.T @ G @ E
rhs = E.T @ G @ target
alpha = np.linalg.solve(H, rhs)
projection_gram = E @ alpha

In [ ]:
print("E =", E, "H =", H, "rhs =", rhs, "alpha =", alpha, "projection_gram =", projection_gram, sep="\n")

**Read the result:** The two bases describe the same approximation space, so both computations give the constant 1/3.

(EᵀGE)α = EᵀG[target]; projection_gram = projection



## Why polynomial geometry needs its own inner product

### An integral can give lengths and angles to functions

The integral is symmetric and bilinear. Also ∫₋₁¹ f(t)² dt ≥ 0. A nonzero polynomial is continuous and cannot vanish throughout an interval; somewhere its squared value is positive on a neighborhood, so the integral is strictly positive. Thus the integral defines an inner product on 𝒫ₙ.

For f(t) = t and g(t) = t², their product t³ is odd. Its negative and positive signed areas cancel, giving inner product 0. This is function-space orthogonality; crossing curves or a right angle between their drawn tangents is not the definition.

⟨t,t²⟩ = 0; ‖t‖² = 2/3; ‖t²‖² = 2/5

### The Gram matrix translates function geometry into coordinates

Expand f = Σcᵢtⁱ and g = Σdⱼtʲ. Bilinearity lets us move the finite sums outside the integral, leaving Σcᵢdⱼ∫tⁱ⁺ʲ = cᵀGd. Symmetry comes from swapping i and j.

For degree two, completing the square gives cᵀGc = 2(c₀+c₂/3)² + (2/3)c₁² + (8/45)c₂². This is positive for every nonzero coefficient vector, so G is positive definite. Coordinate Euclidean length is a different metric.

G = [[2,0,2/3],[0,2/3,0],[2/3,0,2/5]]

### Subtract every earlier basis direction before normalizing

Normalize 1 to q₀ = 1/√2. Because t is odd, it is perpendicular to q₀; normalize it to q₁ = √(3/2)t. The polynomial t² is not perpendicular to the constant: ⟨t²,q₀⟩ = √2/3. Subtracting this component gives w₂(t) = t² − 1/3.

The source loop uses range(i − 1), which skips the immediately preceding basis vector and does no subtraction for i = 1. The maintained routine uses range(j) for all previous vectors with 0-based indexing. It checks the norm before division and verifies CᵀGC, not just the curves’ appearance.

q₂(t) = √(45/8)(t² − 1/3); ∫₋₁¹ (t² − 1/3)² dt = 8/45

### The best line for t² is a constant, by symmetry

To project t² onto 𝒫₁ = span{1,t}, use the orthonormal coordinates ⟨t²,q₀⟩ and ⟨t²,q₁⟩. The latter is zero because t³ is odd. The former times q₀ gives the constant 1/3.

For any line a + bt, the error is the perpendicular residual t² − 1/3 plus a change within 𝒫₁. The integral norm obeys Pythagoras, proving global optimality. The curves need not match at individual points. This is continuous approximation, whereas 4.4 fits a finite list of measurements.

‖t² − (a + bt)‖² = 8/45 + 2(a − 1/3)² + (2/3)b²

### A matrix projection uses the chosen metric

Let E contain coordinates of a basis for the approximation subspace. Writing the approximate coefficient vector as Eα and enforcing EᵀG(c − Eα) = 0 gives (EᵀGE)α = EᵀGc. The metric G belongs in both places.

If the basis columns C are orthonormal for G, α = CᵀGc and the coefficient map is CCᵀG. This reduces to QQᵀ only in a Euclidean orthonormal coordinate system. The same projection idea therefore connects all four labs without silently changing metrics.

G-weighted projection: ĉ = E(EᵀGE)⁻¹EᵀGc

## Numerical checks

These checks verify the example and the correction described above. They support the algebraic arguments and do not replace them.

In [ ]:
assert np.isclose(dot_fg, 0)
assert np.allclose(lengths ** 2, [2 / 3, 2 / 5])
assert np.allclose(G, [[2, 0, 2 / 3], [0, 2 / 3, 0], [2 / 3, 0, 2 / 5]])
assert np.all(np.linalg.eigvalsh(G) > 0)
assert np.allclose(contrast, [0, 2 / 3])
assert np.allclose(w2, [-1 / 3, 0, 1])
assert np.allclose(orthonormal_gram, np.eye(3), rtol=0, atol=1e-12)
assert np.allclose(projection, [1 / 3, 0, 0])
assert np.allclose(orthogonality, 0, rtol=0, atol=1e-12)
assert np.isclose(minimum_error, 8 / 45)
assert np.allclose(projection_gram, projection)
for a, b in [(0, 0), (1 / 3, 0), (-2, 0.5)]:
    candidate = np.array([a, b, 0.])
    error = target - candidate
    assert np.isclose(inner_poly(error, error), 8 / 45 + 2 * (a - 1 / 3) ** 2 + (2 / 3) * b ** 2)

def orthonormal_polynomials(degree):
    """Low-degree teaching routine; coefficient columns use ascending powers."""
    monomials = np.eye(degree + 1)
    accepted = []
    for j in range(degree + 1):
        w = monomials[j].copy()
        # All previous vectors: range(j), never range(j - 1).
        for i in range(j):
            w -= inner_poly(w, accepted[i]) * accepted[i]
        norm_squared = inner_poly(w, w)
        if norm_squared <= 1e-14:
            raise ValueError("Cannot normalize a numerically vanishing polynomial.")
        accepted.append(w / np.sqrt(norm_squared))
    return np.column_stack(accepted)

C3 = orthonormal_polynomials(3)
B3 = np.eye(4)
G3 = np.array([[inner_poly(bi, bj) for bj in B3] for bi in B3])
assert np.allclose(C3.T @ G3 @ C3, np.eye(4), rtol=0, atol=1e-11)
assert np.allclose(C3[:3, :3], C, rtol=0, atol=1e-11)
assert np.allclose(C3[:, 3], np.sqrt(7 / 8) * np.array([0., -3., 0., 5.]))
# Standard Legendre normalization fixes the value at 1, not the integral norm.
P2 = np.array([-0.5, 0., 1.5])
assert np.isclose(poly.polyval(1., P2), 1)
assert np.isclose(inner_poly(P2, P2), 2 / 5)
assert np.allclose(q2, np.sqrt(5 / 2) * P2)
print("All polynomial checks passed.")

## Find the closest cubic to t⁴

Use all four allowed orthonormal directions q₀,…,q₃, exactly as the corrected general projection loop requires. The resulting approximation has degree two by symmetry, but it lies in the space of degree at most three. The graph shows function values; the assertions check the integral metric.

In [ ]:
target4 = np.array([0., 0., 0., 0., 1.])
best_cubic = np.zeros(5)
for j in range(4):
    basis_function = np.pad(C3[:, j], (0, 1))
    best_cubic += inner_poly(target4, basis_function) * basis_function
error4 = target4 - best_cubic
assert np.allclose(best_cubic, [-3 / 35, 0, 6 / 7, 0, 0], rtol=0, atol=1e-11)
assert np.isclose(inner_poly(error4, error4), 128 / 11025)
for j in range(4):
    assert np.isclose(inner_poly(error4, C3[:, j]), 0, atol=1e-11)
print("Closest cubic coefficients =", best_cubic)
import matplotlib.pyplot as plt
t_values = np.linspace(-1, 1, 201)
fig, ax = plt.subplots()
for c, label in [(target4, "t^4"), (best_cubic, "closest polynomial in P3"), (error4, "residual")]:
    ax.plot(t_values, poly.polyval(t_values, c), label=label)
ax.set(xlabel="argument t", ylabel="function value")
ax.legend()
plt.show()

## Check your understanding

**Are sampled heights on a grid the polynomial’s coefficients?**

<details><summary>Show explanation</summary>

No. Coefficients are coordinates in a basis; sampled heights are values f(tₖ). Sampling creates a different representation and a plain sum over samples is not automatically the integral inner product.

</details>

**Why does 1 fail to be perpendicular to t²?**

<details><summary>Show explanation</summary>

Their product t² has integral 2/3 on [−1,1]. Coefficient-array orthogonality does not imply integral orthogonality.

</details>

**Why must the Gram–Schmidt loop use range(j)?**

<details><summary>Show explanation</summary>

At 0-based step j there are j previously accepted basis vectors, with indices 0 through j−1. range(j−1) leaves one out.

</details>

**Is q₂ the standard Legendre polynomial P₂?**

<details><summary>Show explanation</summary>

It is a normalized multiple. P₂(t) = (3t²−1)/2 has P₂(1)=1, while q₂ = √(5/2)P₂ has unit integral norm.

</details>

**Does the best polynomial approximation pass through every target point?**

<details><summary>Show explanation</summary>

No. It minimizes the integrated squared difference over the interval. Here the best line for t² is 1/3, with nonzero error 8/45.

</details>

## Code references

- [NumPy: ascending polynomial coefficients](https://numpy.org/doc/stable/reference/routines.polynomials.polynomial.html)
- [NumPy: polynomial integration](https://numpy.org/doc/stable/reference/generated/numpy.polynomial.polynomial.polyint.html)